# Evaluate Fine-Tuned Embeddings on Eval Pairs

This notebook loads a fine-tuned E5-Mistral checkpoint and evaluates it on labeled story pairs.

Outputs:
- cosine similarity per pair
- ranking metrics (ROC-AUC, Average Precision)
- threshold metrics (accuracy/F1 at best threshold)
- class-wise similarity stats


In [ ]:
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    # Data
    'scores_path': PROJECT_ROOT / 'data' / 'Alignment' / 'asq_random_10000_pair_structural_scores.json',
    'alignments_path': PROJECT_ROOT / 'data' / 'Alignment' / 'asq_random_10000_story_pair_alignments.json',
    'output_dir': PROJECT_ROOT / 'artifacts' / 'e5_mistral_structural_finetune',

    # Model
    'model_name': 'intfloat/e5-mistral-7b-instruct',
    'max_length': 1024,
    'use_bfloat16': True,
    'hf_cache_dir': Path('/scratch/shayan/hf_cache'),

    # Training objective: 'infonce' or 'contrastive_mse'
    'loss_type': 'infonce',

    # Optimization
    'seed': 42,
    'train_frac': 0.9,
    'batch_size': 2,
    'num_epochs': 10,
    'lr': 1e-4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.03,
    'grad_accum_steps': 8,
    'max_grad_norm': 1.0,
    'temperature': 0.05,

    # Pair bucket sampling strategy
    'hard_negative_bin': 2,
    'positive_bin': 3,
    'easy_negative_bins': [0, 1],
    'max_triplets_per_anchor': 8,

    # LoRA
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],

    # Logging
    'eval_every_steps': 100,
}

# User-specified checkpoint
CHECKPOINT_PATH = Path('/scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune/checkpoints/epoch_02')

# Eval set from previous prep
EVAL_PAIRS_PATH = PROJECT_ROOT / 'data' / 'eval_data' / 'eval_story_pairs_200.csv'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if (CONFIG['use_bfloat16'] and DEVICE == 'cuda') else torch.float32

print('PROJECT_ROOT     :', PROJECT_ROOT)
print('EVAL_PAIRS_PATH  :', EVAL_PAIRS_PATH)
print('CHECKPOINT_PATH  :', CHECKPOINT_PATH)
print('DEVICE           :', DEVICE)
print('DTYPE            :', DTYPE)


In [ ]:
if not EVAL_PAIRS_PATH.exists():
    raise FileNotFoundError(f'Eval pairs file not found: {EVAL_PAIRS_PATH}')

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f'Checkpoint path not found: {CHECKPOINT_PATH}\n'
        'Update CHECKPOINT_PATH to your trained checkpoint directory.'
    )

eval_df = pd.read_csv(EVAL_PAIRS_PATH).copy()
required_cols = ['pair_id', 'story_text_a', 'story_text_b', 'label']
missing = [c for c in required_cols if c not in eval_df.columns]
if missing:
    raise ValueError(f'Missing required columns in eval data: {missing}')

eval_df['label'] = eval_df['label'].astype(int)
print('Loaded eval pairs:', len(eval_df))
print(eval_df['label'].value_counts().rename(index={0:'negative',1:'positive'}).to_string())

eval_df.head(3)


In [ ]:
# Load tokenizer + fine-tuned model checkpoint
# This assumes checkpoint directory is in HuggingFace format (e.g. saved via save_pretrained).

tokenizer = AutoTokenizer.from_pretrained(
    str(CHECKPOINT_PATH),
    trust_remote_code=True,
    cache_dir=str(CONFIG['hf_cache_dir']),
)

model = AutoModel.from_pretrained(
    str(CHECKPOINT_PATH),
    trust_remote_code=True,
    cache_dir=str(CONFIG['hf_cache_dir']),
    dtype=DTYPE,
)
model.to(DEVICE)
model.eval()

print('Loaded model from checkpoint:', CHECKPOINT_PATH)


In [ ]:
# E5-style text formatting + mean pooling
# E5 generally benefits from explicit task prefix.

def to_query_text(story: str) -> str:
    return f'query: retrieve stories with a similar narrative to the given story. {story}'


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-9)
    return summed / denom


@torch.no_grad()
def embed_texts(texts, batch_size=8):
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=CONFIG['max_length'],
            return_tensors='pt',
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        out = model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc['attention_mask'])
        pooled = F.normalize(pooled, p=2, dim=1)
        all_emb.append(pooled.detach().cpu())

    return torch.cat(all_emb, dim=0)


In [ ]:
texts_a = [to_query_text(str(x)) for x in eval_df['story_text_a'].tolist()]
texts_b = [to_query_text(str(x)) for x in eval_df['story_text_b'].tolist()]

emb_a = embed_texts(texts_a, batch_size=8)
emb_b = embed_texts(texts_b, batch_size=8)

cos_scores = (emb_a * emb_b).sum(dim=1).numpy()
eval_df['cosine_similarity'] = cos_scores

print('Computed cosine scores for', len(eval_df), 'pairs')
eval_df[['pair_id', 'label', 'cosine_similarity']].head(10)


In [ ]:
y_true = eval_df['label'].to_numpy(dtype=int)
y_score = eval_df['cosine_similarity'].to_numpy(dtype=float)

metrics = {}

# Ranking metrics
metrics['roc_auc'] = float(roc_auc_score(y_true, y_score))
metrics['average_precision'] = float(average_precision_score(y_true, y_score))

# Best-threshold metrics (maximize F1)
thresholds = np.unique(y_score)
best = {'threshold': None, 'f1': -1.0, 'accuracy': None, 'precision': None, 'recall': None}

for t in thresholds:
    y_pred = (y_score >= t).astype(int)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    if f1 > best['f1']:
        p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
        acc = accuracy_score(y_true, y_pred)
        best.update({'threshold': float(t), 'f1': float(f1), 'accuracy': float(acc), 'precision': float(p), 'recall': float(r)})

metrics['best_threshold'] = best

# Class-wise score stats
by_label = eval_df.groupby('label')['cosine_similarity'].agg(['count', 'mean', 'std', 'min', 'max'])
by_label.index = by_label.index.map({0: 'negative', 1: 'positive'})

print('=== Ranking Metrics ===')
print(json.dumps({'roc_auc': metrics['roc_auc'], 'average_precision': metrics['average_precision']}, indent=2))

print('\n=== Best Threshold Metrics (max F1) ===')
print(json.dumps(best, indent=2))

print('\n=== Cosine Similarity by Class ===')
print(by_label.round(6).to_string())


In [ ]:
# Optional: persist per-pair scores + aggregate metrics
OUT_DIR = PROJECT_ROOT / 'data' / 'eval_results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

pair_scores_path = OUT_DIR / 'embedding_eval_pair_scores.csv'
metrics_path = OUT_DIR / 'embedding_eval_metrics.json'

eval_df.to_csv(pair_scores_path, index=False)
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(
        {
            'checkpoint_path': str(CHECKPOINT_PATH),
            'model_name': CONFIG['model_name'],
            'loss_type': CONFIG['loss_type'],
            'num_pairs': int(len(eval_df)),
            'num_positive': int((eval_df['label'] == 1).sum()),
            'num_negative': int((eval_df['label'] == 0).sum()),
            'roc_auc': metrics['roc_auc'],
            'average_precision': metrics['average_precision'],
            'best_threshold': metrics['best_threshold'],
        },
        f,
        indent=2,
    )

print('Saved pair scores:', pair_scores_path)
print('Saved metrics    :', metrics_path)
